## Entity Recognition & Key Phrase Extraction

### 학습 목표
1. `detect_entities`로 텍스트에서 인물·장소·조직·날짜 등을 추출한다.
2. `detect_key_phrases`로 문서의 핵심 명사구를 추출한다.
3. 추출 결과를 시각화하고 DataFrame으로 정리한다.

### API 개요
```python
# 개체 인식
comprehend.detect_entities(Text='...', LanguageCode='ko')

# 핵심 문구 추출
comprehend.detect_key_phrases(Text='...', LanguageCode='ko')
```

### Entity 타입
```
PERSON  LOCATION  ORGANIZATION  DATE  QUANTITY
TITLE   EVENT     COMMERCIAL_ITEM  OTHER
```

### Entity 결과 구조
```
{
  'Text': 'Amazon',
  'Type': 'ORGANIZATION',
  'Score': 0.998,
  'BeginOffset': 0,
  'EndOffset': 6
}
```


In [ ]:
# ✅ [제공 코드] 환경 초기화
import boto3
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
from collections import Counter

# 한글 폰트 설정 (SageMaker Studio) — 최초 1회만 설치, 이후 즉시 로드
try:
    import koreanize_matplotlib
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'koreanize-matplotlib'])
    import koreanize_matplotlib

comprehend = boto3.client('comprehend', region_name='ap-northeast-2')
print('✅ Comprehend 클라이언트 생성 완료 (서울 ap-northeast-2)')


In [ ]:
#detect_entities

news_text = """
2024년 3월 15일, 삼성전자는 서울 강남구 삼성 서초사옥에서 갤럭시 S25를 공식 출시했습니다.
CEO 이재용 부회장이 직접 발표를 진행했으며, 애플의 아이폰 15와의 경쟁에서
우위를 점할 것으로 기대된다고 밝혔습니다. 출시 행사에는 약 500명의 기자가 참석했습니다.
"""

response = comprehend.detect_entities(
    Text=news_text,          
    LanguageCode='ko'  
)

entities = response['Entities'] 
print(f'감지된 개체: {len(entities)}개')
print('-' * 60)
for e in entities:
    print(f"  [{e['Type']:<20}] {e['Text']:<20} (신뢰도: {e['Score']:.3f})") 


In [ ]:
#개체 유형별 시각화
type_colors = {
    'PERSON':'#3498db', 'LOCATION':'#2ecc71', 'ORGANIZATION':'#e74c3c',
    'DATE':'#f39c12', 'QUANTITY':'#9b59b6', 'COMMERCIAL_ITEM':'#1abc9c',
    'OTHER':'#95a5a6', 'TITLE':'#e67e22', 'EVENT':'#34495e'
}
type_counts = Counter(e['Type'] for e in entities)
labels = list(type_counts.keys())
sizes  = list(type_counts.values())
colors = [type_colors.get(l, '#95a5a6') for l in labels]

plt.figure(figsize=(8,4))
plt.bar(labels, sizes, color=colors)
plt.title('개체 유형별 분포')
plt.ylabel('개수')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
#detect_key_phrases

article = """
인공지능 기술의 급속한 발전으로 의료 분야에서 혁신적인 변화가 일어나고 있습니다.
딥러닝 기반 진단 시스템은 암 조기 발견 정확도를 90% 이상으로 높였으며,
자연어처리 기술을 활용한 의료 챗봇은 환자 상담 서비스를 24시간 제공합니다.
"""

response = comprehend.detect_key_phrases(
    Text=article,    
    LanguageCode='ko'
)

phrases = response['KeyPhrases'] 

# 신뢰도 기준 내림차순 정렬
phrases_sorted = sorted(phrases, key=lambda x: x['Score'], reverse=True)

print(f'추출된 핵심 문구: {len(phrases)}개')
print('-' * 50)
for p in phrases_sorted[:10]:
    print(f"  {p['Score']:.3f}  {p['Text']}")


In [ ]:
#개체 & 핵심문구 통합 분석
combined_text = """
구글은 2023년 12월 뉴욕에서 Gemini Pro를 발표했습니다.
이는 ChatGPT에 대응하는 구글의 핵심 AI 전략으로,
클라우드 서비스와 검색 엔진에 통합될 예정입니다.
"""

# 두 API 동시 호출
ent_resp = comprehend.detect_entities(Text=combined_text, LanguageCode='ko')
kp_resp  = comprehend.detect_key_phrases(Text=combined_text, LanguageCode='ko')

# 개체 DataFrame
df_ent = pd.DataFrame([
    {'구분':'개체', '텍스트': e['Text'], '유형': e['Type'], '신뢰도': round(e['Score'],3)} 
    for e in ent_resp['Entities']
])

# 핵심문구 DataFrame
df_kp = pd.DataFrame([
    {'구분':'핵심문구', '텍스트': p['Text'], '유형': '-', '신뢰도': round(p['Score'],3)}
    for p in kp_resp['KeyPhrases']
])

df_all = pd.concat([df_ent, df_kp], ignore_index=True)
df_all.to_csv('lab02_result.csv', index=False, encoding='utf-8-sig')
print(df_all.to_string(index=False))
print('\n✅ lab02_result.csv 저장 완료')
